# Auto-Quant - GPTQ Quantization (Kaggle T4)

Run this notebook on Kaggle with **GPU T4 x2** accelerator enabled.

**Before running:**
1. Kaggle -> Settings -> Accelerator -> GPU T4 x2
2. Set your HuggingFace token in `HF_TOKEN` below if the model is gated
3. Set `MODEL_ID` and `GPTQ_BITS` as needed
4. After running the install cell, go to Run -> Restart & Run All

**Output:** Quantized model saved to `/kaggle/working/gptq_output/` - download from the Data tab when complete.

In [ ]:
# -- Configuration -------------------------------------------------------------
MODEL_ID       = "Qwen/Qwen2-0.5B"   # HuggingFace repo ID
GPTQ_BITS      = [4, 8]              # Which bit levels to quantize
GROUP_SIZE     = 128                 # GPTQ group size - 128 is standard
HF_TOKEN       = None                # Set to your HF token if model is gated
OUTPUT_DIR     = "/kaggle/working/gptq_output"
# ------------------------------------------------------------------------------

In [ ]:
import subprocess, os

subprocess.run([
    "pip", "install", "-q",
    "gptqmodel",
    "transformers==4.51.3",
    "datasets",
    "safetensors",
    "accelerate",
], check=True)

print("Install complete. Now go to Run -> Restart & Run All to reload with clean kernel.")
print("Skip this cell on the second run by commenting out the subprocess call.")

In [ ]:
import torch
import numpy
import gptqmodel

assert torch.cuda.is_available(), "No GPU found - enable T4 in Kaggle Settings > Accelerator"
print(f"numpy: {numpy.__version__}")
print(f"torch: {torch.__version__}")
print(f"gptqmodel: {gptqmodel.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
from gptqmodel import GPTQModel, QuantizeConfig
from transformers import AutoTokenizer
from datasets import load_dataset
from pathlib import Path
import torch, gc, json

results = {}

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

print("Loading calibration dataset (WikiText-2)...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
samples = [
    sample["text"]
    for sample in dataset.select(range(256))
    if len(sample["text"].strip()) > 0
][:128]
print(f"Calibration samples ready: {len(samples)}")

for bits in GPTQ_BITS:
    level = f"int{bits}"
    output_path = Path(OUTPUT_DIR) / level
    output_path.mkdir(parents=True, exist_ok=True)

    print(f"\n-- Quantizing to {level} --")

    quant_config = QuantizeConfig(
        bits=bits,
        group_size=GROUP_SIZE,
    )

    print("Loading model...")
    model = GPTQModel.load(
        MODEL_ID,
        quant_config,
        token=HF_TOKEN,
        trust_remote_code=True,
    )

    print("Running calibration and quantization...")
    model.quantize(samples, tokenizer=tokenizer)

    print(f"Saving to {output_path}...")
    model.save(str(output_path))
    tokenizer.save_pretrained(str(output_path))

    meta = {
        "model_id": MODEL_ID,
        "level": level,
        "bits": bits,
        "group_size": GROUP_SIZE,
        "calibration_dataset": "wikitext-2-raw-v1",
        "backend": "gptqmodel",
    }
    with open(output_path / "autoquant_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    size_mb = sum(
        p.stat().st_size for p in output_path.rglob("*") if p.is_file()
    ) / (1024 * 1024)
    print(f"Done: {output_path} ({size_mb:.1f} MB)")
    results[level] = str(output_path)

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll levels complete:", results)

In [ ]:
import os
print("Output files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        full = os.path.join(root, f)
        size_mb = os.path.getsize(full) / (1024 * 1024)
        print(f"  {full} ({size_mb:.1f} MB)")

print("\nDownload from: Kaggle > notebook > Output tab > gptq_output/")